# 06. 최소제곱법 (Least Squares)

$$\min_x \|Ax - b\|^2 \implies x^* = (A^T A)^{-1} A^T b$$

방정식이 미지수보다 많을 때 (overdetermined) 정확한 해는 없다.
대신 **잔차의 제곱합을 최소화**하는 해를 구한다.

**로보틱스 연결:**
- 센서 캘리브레이션 — 측정값 N개로 파라미터 M개 추정 (N >> M)
- SLAM 포즈 그래프 최적화 — 루프 클로저 포함한 과결정 시스템
- 카메라 내부 파라미터 추정 (Zhang's method)
- 야코비안 기반 역기구학 — $\dot{q} = J^+ \dot{x}$ (의사역행렬 = 최소제곱 해)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs('assets', exist_ok=True)

plt.rcParams['font.family'] = 'Nanum Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 문제 설정 — 노이즈 낀 센서 데이터에 직선 피팅

로봇 관절 센서 30개 측정값 → 직선 $y = ax + b$ 피팅.
행렬로 쓰면:

$$\begin{bmatrix} x_1 & 1 \\ x_2 & 1 \\ \vdots & \vdots \\ x_N & 1 \end{bmatrix} \begin{bmatrix} a \\ b \end{bmatrix} = \begin{bmatrix} y_1 \\ y_2 \\ \vdots \\ y_N \end{bmatrix}$$

$N=30$, 미지수 2개 → overdetermined.

In [ ]:
np.random.seed(7)
n = 30
x_data = np.linspace(0, 10, n)
a_true, b_true = 2.5, 1.3
y_data = a_true * x_data + b_true + np.random.randn(n) * 2.0

A = np.column_stack([x_data, np.ones(n)])
b_vec = y_data

# 정규 방정식: (A^T A) x = A^T b
ATA = A.T @ A
ATb = A.T @ b_vec
x_normal = np.linalg.solve(ATA, ATb)

# numpy 내장
x_lstsq, residuals, rank, sv = np.linalg.lstsq(A, b_vec, rcond=None)

print(f'실제 파라미터:        a={a_true}, b={b_true}')
print(f'정규 방정식 결과:     a={x_normal[0]:.4f}, b={x_normal[1]:.4f}')
print(f'np.linalg.lstsq:     a={x_lstsq[0]:.4f}, b={x_lstsq[1]:.4f}')
print(f'잔차 제곱합: {np.sum((A @ x_normal - b_vec)**2):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(x_data, y_data, color='#534AB7', s=35, alpha=0.7, label='측정값 (노이즈 포함)')
x_plot = np.linspace(0, 10, 100)
ax.plot(x_plot, a_true*x_plot + b_true, 'k--', lw=1.5, alpha=0.5, label=f'실제 y={a_true}x+{b_true}')
ax.plot(x_plot, x_normal[0]*x_plot + x_normal[1], color='#E85D24', lw=2.5,
        label=f'최소제곱 y={x_normal[0]:.2f}x+{x_normal[1]:.2f}')

# 잔차 시각화
for xi, yi in zip(x_data, y_data):
    y_hat = x_normal[0]*xi + x_normal[1]
    ax.plot([xi, xi], [yi, y_hat], color='gray', lw=0.8, alpha=0.5)

ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('최소제곱 직선 피팅\n(회색 선 = 잔차)', fontsize=11)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# 잔차 분포
ax2 = axes[1]
residuals_vec = b_vec - A @ x_normal
ax2.hist(residuals_vec, bins=12, color='#534AB7', alpha=0.7, edgecolor='white')
ax2.axvline(0, color='#E85D24', lw=2, linestyle='--')
ax2.set_xlabel('잔차'); ax2.set_ylabel('빈도')
ax2.set_title(f'잔차 분포\n평균={residuals_vec.mean():.4f}, std={residuals_vec.std():.4f}', fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('assets/06_least_squares_fit.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. 기하학적 의미 — 열 공간으로의 정사영

$Ax$ 가 만들 수 있는 벡터의 집합 = 열 공간 $\text{Col}(A)$.
$b$ 가 이 공간에 없으면 → $b$를 열 공간에 **정사영**한 $\hat{b} = Ax^*$ 가 최선.
오차 $r = b - \hat{b}$ 는 열 공간과 수직.

$$A^T(b - Ax^*) = 0 \implies A^T A x^* = A^T b$$

이게 정규 방정식이 나오는 이유.

In [ ]:
# 3D에서 2D 열 공간으로 정사영 시각화
np.random.seed(3)
A3 = np.array([[1., 0.], [0., 1.], [0.5, 0.5]])  # 3x2 — 열 공간은 R^3의 2D 평면
b3 = np.array([1.5, 0.8, 1.8])

x3 = np.linalg.lstsq(A3, b3, rcond=None)[0]
b3_hat = A3 @ x3
r3 = b3 - b3_hat

fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection='3d')

# 열 공간 (평면)
s, t = np.meshgrid(np.linspace(-1, 2.5, 10), np.linspace(-1, 2.5, 10))
plane = np.einsum('ij,k->ijk', s, A3[:,0]) + np.einsum('ij,k->ijk', t, A3[:,1])
ax.plot_surface(plane[:,:,0], plane[:,:,1], plane[:,:,2],
                alpha=0.15, color='#534AB7')

# b, b_hat, 잔차
ax.quiver(0,0,0, *b3, color='#E85D24', lw=2.5, arrow_length_ratio=0.1, label='b')
ax.quiver(0,0,0, *b3_hat, color='#1D9E75', lw=2.5, arrow_length_ratio=0.1, label='Ax* (정사영)')
ax.quiver(*b3_hat, *r3, color='gray', lw=2, arrow_length_ratio=0.1, linestyle='dashed', label='r = b-Ax*')

# 직각 표시
ax.text(*b3*1.05, 'b', fontsize=12, color='#E85D24', fontweight='bold')
ax.text(*b3_hat*1.05, 'Ax*', fontsize=12, color='#1D9E75', fontweight='bold')

ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('열 공간으로의 정사영\nb → Ax* (잔차 r ⊥ 열 공간)', fontsize=11, pad=10)
ax.legend(fontsize=9)

print(f'r · col1(A) = {np.dot(r3, A3[:,0]):.2e}  (≈0 이어야 함)')
print(f'r · col2(A) = {np.dot(r3, A3[:,1]):.2e}  (≈0 이어야 함)')
plt.tight_layout()
plt.show()

## 3. QR 분해 — 수치적으로 더 안정한 풀이

정규 방정식 $(A^T A)^{-1}$ 를 직접 계산하면 $A^T A$ 의 조건수가 $A$ 의 **제곱**이 된다.
$A = QR$ 로 분해하면 $Rx = Q^T b$ 를 풀면 되고 조건수가 훨씬 작다.

실제 `np.linalg.lstsq` 내부도 QR/SVD를 쓴다.

In [ ]:
# 조건수 비교
np.random.seed(42)
n_cond = 20
x_c = np.linspace(0, 1, n_cond)
A_cond = np.column_stack([x_c**i for i in range(6)])  # 다항식 피팅 (ill-conditioned)

cond_A   = np.linalg.cond(A_cond)
cond_ATA = np.linalg.cond(A_cond.T @ A_cond)
print(f'cond(A)   = {cond_A:.2e}')
print(f'cond(AᵀA) = {cond_ATA:.2e}  ← 제곱으로 커짐')

# QR 방법
Q, R = np.linalg.qr(A_cond)
b_c = np.random.randn(n_cond)

x_qr     = np.linalg.solve(R, Q.T @ b_c)
x_normal2 = np.linalg.solve(A_cond.T @ A_cond, A_cond.T @ b_c)
x_lstsq2, _, _, _ = np.linalg.lstsq(A_cond, b_c, rcond=None)

print(f'\n잔차 비교:')
print(f'정규 방정식: {np.linalg.norm(A_cond @ x_normal2 - b_c):.6f}')
print(f'QR 분해:    {np.linalg.norm(A_cond @ x_qr - b_c):.6f}')
print(f'lstsq:     {np.linalg.norm(A_cond @ x_lstsq2 - b_c):.6f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
methods_c = ['정규 방정식\n(AᵀA)⁻¹Aᵀb', 'QR 분해', 'np.lstsq']
errors_c  = [
    np.linalg.norm(A_cond @ x_normal2 - b_c),
    np.linalg.norm(A_cond @ x_qr - b_c),
    np.linalg.norm(A_cond @ x_lstsq2 - b_c),
]
bars_c = ax.bar(methods_c, errors_c, color=['#E85D24','#1D9E75','#534AB7'], width=0.4)
for bar, val in zip(bars_c, errors_c):
    ax.text(bar.get_x()+bar.get_width()/2, val*1.05, f'{val:.4f}',
            ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('잔차 크기'); ax.set_title('풀이 방법별 잔차 비교\n(ill-conditioned 행렬)', fontsize=11)
ax.grid(True, axis='y', alpha=0.3)

ax2 = axes[1]
cond_vals = [np.linalg.cond(np.column_stack([x_c**i for i in range(k)])) for k in range(2, 9)]
ax2.semilogy(range(2, 9), cond_vals, 'o-', color='#534AB7', lw=2, markersize=6)
ax2.semilogy(range(2, 9), [c**2 for c in cond_vals], 's--', color='#E85D24', lw=1.5,
             markersize=6, alpha=0.6, label='(이론) cond(AᵀA) ≈ cond(A)²')
ax2.set_xlabel('다항식 차수'); ax2.set_ylabel('조건수 (log)')
ax2.set_title('조건수 vs 다항식 차수\ncond(AᵀA) ≈ cond(A)²', fontsize=11)
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('assets/06_condition_number.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 로보틱스 응용 — 가속도계 캘리브레이션

이상적인 가속도계: $\|a_{measured}\| = g = 9.81$.
실제로는 바이어스(bias)와 스케일 오차가 섞임:

$$a_{true} = S \cdot a_{measured} + b_{bias}$$

여러 자세에서 측정 후 최소제곱으로 $S$, $b$ 추정.

In [ ]:
np.random.seed(0)
n_meas = 40

# 다양한 방향에서 중력 벡터 측정 (이상: 크기 = 9.81)
directions = np.random.randn(n_meas, 3)
directions /= np.linalg.norm(directions, axis=1, keepdims=True)
g = 9.81

# 실제 센서: 스케일 오차 + 바이어스 + 노이즈
S_true   = np.array([1.05, 0.97, 1.02])  # 각 축 스케일
bias_true = np.array([0.15, -0.08, 0.22])
noise_std = 0.05

measurements = directions * g * S_true + bias_true + np.random.randn(n_meas, 3)*noise_std

# 최소제곱으로 [Sx, Sy, Sz, bx, by, bz] 추정
# 각 측정에서 ||S*m + b||^2 = g^2 → 비선형
# 선형화: 각 축 독립 피팅 (Sx*mx + bx ≈ g*dx)
results = {}
for axis_idx, axis_name in enumerate(['X', 'Y', 'Z']):
    A_cal = np.column_stack([measurements[:, axis_idx], np.ones(n_meas)])
    b_cal = directions[:, axis_idx] * g
    x_cal = np.linalg.lstsq(A_cal, b_cal, rcond=None)[0]
    results[axis_name] = {'scale': x_cal[0], 'bias': x_cal[1]}

print('캘리브레이션 결과:')
print(f"{'축':>4} | {'실제 스케일':>10} | {'추정 스케일':>10} | {'실제 바이어스':>12} | {'추정 바이어스':>12}")
print('-' * 62)
for i, (axis, s_true, b_true_val) in enumerate(zip(['X','Y','Z'], S_true, bias_true)):
    r = results[axis]
    print(f"  {axis}  | {s_true:>10.4f} | {r['scale']:>10.4f} | {b_true_val:>12.4f} | {r['bias']:>12.4f}")

# 캘리브레이션 전후 크기 오차
raw_norms = np.linalg.norm(measurements, axis=1)
cal_meas = np.zeros_like(measurements)
for i, axis in enumerate(['X','Y','Z']):
    r = results[axis]
    cal_meas[:, i] = (measurements[:, i] - r['bias']) / r['scale']
cal_norms = np.linalg.norm(cal_meas, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, norms, title, color in [
    (axes[0], raw_norms, '캘리브레이션 전\n||a|| 분포', '#E85D24'),
    (axes[1], cal_norms, '캘리브레이션 후\n||a|| 분포', '#1D9E75')
]:
    ax.hist(norms, bins=15, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(g, color='k', lw=2, linestyle='--', label=f'이상값 g={g}')
    ax.axvline(norms.mean(), color=color, lw=1.5, linestyle='-',
               label=f'평균={norms.mean():.3f}')
    ax.set_xlabel('||a|| (m/s²)'); ax.set_ylabel('빈도')
    ax.set_title(f'{title}\n(std={norms.std():.4f})', fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('assets/06_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## 요약

| 방법 | 수식 | 언제 쓰나 |
|------|------|-----------|
| 정규 방정식 | $(A^T A)^{-1} A^T b$ | 조건수 작을 때 |
| QR 분해 | $A=QR,\ Rx=Q^Tb$ | 일반적인 경우 |
| SVD 기반 | $A^+ = V\Sigma^+ U^T$ | 랭크 부족 행렬 |
| `np.linalg.lstsq` | 내부적으로 SVD/QR | 실용적으로 그냥 이거 쓰면 됨 |

**선형대수 파트 끝.** 다음은 `02_calculus/` — 미적분, 그라디언트, 자코비안.